# 第5章 LangGraph + LangSmith

LangChain の `AgentExecutor` の発展形である **LangGraph** と、実行トレースを可視化する **LangSmith** を体験します。

**所要時間**: 90〜120分

## 学ぶこと
1. prebuilt の `create_react_agent` で 1 行エージェントを作る
2. 手書きの `StateGraph` でノード/エッジを定義する
3. LangSmith にトレースを送り、エージェント内部の動きをダッシュボードで確認する
4. `MemorySaver` で会話履歴を保持し、複数ターンの対話を成立させる

## なぜ Agent から LangGraph に進むのか

第3章で見た `AgentExecutor` は便利ですが、内部はブラックボックスで以下のような要件には窮屈です:

- **分岐**: 「ツール結果がXならYに進む、ZならWに進む」
- **状態管理**: 「会話履歴」「中間結果」を明示的に持ちたい
- **人間レビュー**: 途中でユーザーの確認を挟みたい
- **並列実行**: 複数の処理を同時に走らせたい
- **再開可能性**: 途中で止めて、後から再開したい

LangGraph は **「ノード(処理)」と「エッジ(遷移)」のグラフ** としてエージェントを記述するライブラリです。
状態(State)を明示的に持つので、上記のような複雑な制御が書きやすくなります。

```
AgentExecutor の世界          LangGraph の世界
  ┌────────────┐              ┌──────────┐
  │ LLM ⇄ Tool │      →       │ State    │
  │ ループ      │              │ Node1 → Node2 → Node3
  └────────────┘              │ 分岐・並列・記憶可
                              └──────────┘
```

## 0. 準備 - 環境変数の読み込みと LangSmith 設定確認

In [2]:
import os
from dotenv import load_dotenv

# .env を読み込む。LANGSMITH_TRACING=true / LANGSMITH_API_KEY も読まれる
# 重要: LangSmith は環境変数を見て「自動的に」トレースを送る。明示的なコードは不要
load_dotenv()

AWS_REGION = os.environ["AWS_REGION"]
CHAT_MODEL_ID = os.environ["BEDROCK_CHAT_MODEL_ID"]

# LangSmith 設定の状況を確認
print("region:", AWS_REGION)
print("chat model:", CHAT_MODEL_ID)
print("LANGSMITH_TRACING:", os.environ.get("LANGSMITH_TRACING", "(unset)"))
print("LANGSMITH_PROJECT:", os.environ.get("LANGSMITH_PROJECT", "(unset)"))
print("LANGSMITH_ENDPOINT:", os.environ.get("LANGSMITH_ENDPOINT", "(unset)"))
print("LANGSMITH_API_KEY:",
      "set (length=%d)" % len(os.environ["LANGSMITH_API_KEY"])
      if os.environ.get("LANGSMITH_API_KEY") else "(unset - トレースは送信されません)")

region: ap-northeast-1
chat model: apac.amazon.nova-lite-v1:0
LANGSMITH_TRACING: true
LANGSMITH_PROJECT: work-project
LANGSMITH_ENDPOINT: https://apac.api.smith.langchain.com
LANGSMITH_API_KEY: set (length=51)


**API キーがまだ未設定なら**: 別途 [LangSmithセットアップガイド](../docs/05_langsmith_setup.md) を実施してください。
未設定のままでも Notebook は動きますが、トレース可視化のステップはスキップになります。

## 1. LLM とツールの準備

第3章と同じ2つのツールを用意します。LangGraph では `@tool` でツール化したものをそのまま使えます。

In [ ]:
from datetime import datetime
from zoneinfo import ZoneInfo
from langchain_aws import ChatBedrockConverse
from langchain_core.tools import tool

# LLM を準備(第1章と同じ。LangGraph でも LLM オブジェクトはそのまま使い回せる)
llm = ChatBedrockConverse(
    model=CHAT_MODEL_ID,
    region_name=AWS_REGION,
    temperature=0,
)

# --- ツール定義(第3章のおさらい) ---
# @tool デコレータを付けると、ただの関数が「LLMが呼べるツール」に変換される。
# docstring がツールの説明として、型ヒント(timezone: str など)が引数スキーマとして LLM に渡る。
@tool
def get_current_time(timezone: str = "Asia/Tokyo") -> str:
    """指定したタイムゾーンの現在時刻を ISO 8601 文字列で返す。"""
    return datetime.now(ZoneInfo(timezone)).isoformat(timespec="seconds")

@tool
def add_numbers(a: float, b: float) -> float:
    """2つの数値を加算した結果を返す。"""
    return a + b

# 複数のツールはリストにまとめて、エージェントやグラフに渡す
tools = [get_current_time, add_numbers]

## 2. prebuilt の `create_react_agent` で 1 行エージェント

LangGraph には **「典型的な ReAct パターン(推論↔行動のループ)」をすぐ作れる prebuilt 関数** が用意されています。
第3章の `create_tool_calling_agent` + `AgentExecutor` の LangGraph 版と考えてください。

**戻り値**: コンパイル済みグラフ。`.invoke()` / `.stream()` できる Runnable。
**入力形式**: `{"messages": [{"role": "user", "content": "..."}]}` というメッセージのリストを渡す。

In [ ]:
from langgraph.prebuilt import create_react_agent

# create_react_agent(モデル, ツール) だけでエージェントが完成する。
# 内部では「LLM呼び出し → ツール実行 → LLM呼び出し → ...」のループ(ReActパターン)が
# グラフとして自動で組まれる。第3章の create_tool_calling_agent + AgentExecutor 相当。
agent = create_react_agent(llm, tools)

# 入力は {"messages": [...]} 形式。各メッセージは {"role": ..., "content": ...} の dict。
# role は "user"(ユーザー) / "assistant"(AI) / "system"(指示) など。OpenAI 互換の形式。
result = agent.invoke({
    "messages": [{"role": "user", "content": "いま東京は何時? それと 12.5 と 7.5 を足して。"}]
})

# 戻り値の result も {"messages": [...]} 形式。
# messages には [人間の質問, AIのツール呼び出し, ツール結果, ..., 最終回答] が時系列で全部入る。
# m.pretty_print() は各メッセージを役割(Human/Ai/Tool)つきで見やすく表示するヘルパー。
print("=== 全メッセージ履歴 ===")
for m in result["messages"]:
    m.pretty_print()

## 3. LangSmith ダッシュボードで実行を確認

`.env` に `LANGSMITH_API_KEY` を設定していれば、上のセル実行が **自動的に LangSmith に送信されている** はずです。

**確認手順:**
1. https://smith.langchain.com/ にログイン
2. 左メニュー「Tracing Projects」 → `langchain-bedrock-handson`(`.env` の `LANGSMITH_PROJECT` 名)
3. 直近のトレースをクリック
4. ツリーを展開すると、以下のような階層が見えるはず:
   ```
   create_react_agent
   ├─ ChatBedrockConverse (1回目)         ← LLMが「ツールを呼ぶ」と判断
   ├─ get_current_time                   ← ツール実行
   ├─ ChatBedrockConverse (2回目)         ← 結果を見て「次もツール」と判断
   ├─ add_numbers                        ← ツール実行
   └─ ChatBedrockConverse (3回目)         ← 結果をまとめて最終回答
   ```
5. 各ノードをクリックすると **送信したプロンプト・受信した出力・トークン消費・所要時間** が見られる

verbose ログより圧倒的にデバッグしやすいですね。これが LangSmith の威力です。

## 4. 手書きの StateGraph で同じものを組む

prebuilt は便利ですが、自分でカスタマイズしたくなった時のために **StateGraph の基本** を理解しておきましょう。

### 基本要素

| 要素 | 役割 |
|---|---|
| **State** | グラフ全体で共有する状態(辞書のような型)|
| **Node** | 状態を受け取り、状態の更新を返す関数 |
| **Edge** | ノード間の遷移。条件付き分岐も書ける |
| **`START` / `END`** | グラフの入り口/出口の特殊ノード |

### 今回作るグラフ

```
      START
        ↓
    call_model ←──┐
        ↓         │
  tools_condition │   (ツール呼び出しがあれば tools へ)
      ↓     ↓     │
    END    tools ─┘   (なければ END)
```

ReAct パターンの最小構成です。LLMノード(`call_model`)とツール実行ノード(`tools`)を、条件付きエッジで結ぶだけ。

In [ ]:
from langgraph.graph import StateGraph, START, END, MessagesState
from langgraph.prebuilt import ToolNode, tools_condition

# === State(状態)とは ===
# グラフの全ノードで共有される「作業メモ」のようなもの。各ノードはこれを読んで、更新を返す。
# MessagesState は LangGraph 組み込みの State で、{"messages": [...]} という1キーだけを持つ。
# 特別な性質: ノードが {"messages": [新メッセージ]} を返すと、上書きではなく「追記」される。

# === LLM にツールを bind する ===
# .bind_tools(tools) で「このLLMはこれらのツールを呼べる」と教える。
# これにより、LLM の応答に tool_calls(どのツールをどんな引数で呼ぶか)が含まれるようになる。
llm_with_tools = llm.bind_tools(tools)

# === ノード = 「状態を受け取り、更新分を返す」関数 ===
# call_model は「現在の会話履歴を LLM に渡し、その応答を履歴に足す」だけのノード。
def call_model(state: MessagesState):
    response = llm_with_tools.invoke(state["messages"])  # 履歴全体を渡す
    return {"messages": [response]}  # 返り値は「更新分」。MessagesState なので自動で追記される

# === グラフの組み立て ===
builder = StateGraph(MessagesState)  # State の型を指定してビルダーを作る

# add_node(ノード名, 関数): ノードを登録する
builder.add_node("call_model", call_model)   # 自作の LLM ノード
builder.add_node("tools", ToolNode(tools))   # ツール実行は prebuilt の ToolNode に任せる

# add_edge(始点, 終点): 無条件の遷移。開始したらまず call_model へ
builder.add_edge(START, "call_model")

# add_conditional_edges(始点, 判定関数): 条件分岐の遷移。
# tools_condition は「直前の LLM 応答に tool_calls があるか」を判定する prebuilt 関数:
#   - ツール呼び出しあり → "tools" ノードへ
#   - なし(=最終回答)  → END へ
builder.add_conditional_edges("call_model", tools_condition)

# ツール実行が終わったら call_model に戻る(結果を見て LLM が次を判断 → ループ)
builder.add_edge("tools", "call_model")

# compile() で実行可能なグラフ(Runnable)に変換。これで .invoke() などが使える
graph = builder.compile()

# 実行。入力形式は create_react_agent と同じ {"messages": [...]}
result = graph.invoke({
    "messages": [{"role": "user", "content": "今 UTC で何時? その後 3.14 と 2.86 を足して。"}]
})

print("=== 全メッセージ履歴 ===")
for m in result["messages"]:
    m.pretty_print()

## 5. グラフを可視化する

組み上げたグラフの構造をテキスト(ASCII)で確認できます。

Mermaid 形式の出力も可能で、Markdown ビューアやドキュメントに貼り付けて使えます。

In [ ]:
# get_graph() でグラフ構造を取り出し、draw_mermaid() で Mermaid 記法の文字列に変換する。
# 出力を GitHub の Markdown や https://mermaid.live に貼ると、図として描画できる。
# __start__ → call_model、条件分岐で tools か __end__、tools からは call_model に戻る、が読み取れる。
print(graph.get_graph().draw_mermaid())

In [7]:
# ASCII 形式での可視化(grandalf パッケージが入っていれば動作)
try:
    print(graph.get_graph().draw_ascii())
except Exception as e:
    print("ASCII 描画には grandalf が必要です。Mermaid 出力(上のセル)を使ってください。")
    print(f"  詳細: {e}")

        +-----------+         
        | __start__ |         
        +-----------+         
               *              
               *              
               *              
        +------------+        
        | call_model |        
        +------------+        
          .         *         
        ..           **       
       .               *      
+---------+         +-------+ 
| __end__ |         | tools | 
+---------+         +-------+ 


## 6. MemorySaver で会話履歴を保持する

ここまでは「1回の質問→回答」で完結していました。実用的なチャットアプリでは **過去の会話を覚えていてほしい** ですよね。

LangGraph は **Checkpointer** という仕組みで、各実行のステートをスナップショット保存できます。
`MemorySaver` はその最も簡単な実装(プロセスメモリに保存)。

**使い方**:
1. グラフを `compile(checkpointer=memory)` で再コンパイル
2. 実行時に `config={"configurable": {"thread_id": "任意のID"}}` を渡す
3. 同じ `thread_id` で次の invoke を呼ぶと、前回の messages が自動で復元される

本番では `SqliteSaver` / `PostgresSaver` 等の永続化版に差し替えるだけで OK。

In [ ]:
from langgraph.checkpoint.memory import MemorySaver

# Checkpointer: 各ターンの State(会話履歴)を保存しておく仕組み。
# MemorySaver はプロセスのメモリ上に保存する最も簡単な実装(カーネル再起動で消える)。
memory = MemorySaver()

# 同じ builder(グラフの設計図)から、checkpointer 付きで再コンパイルする。
# これで「実行のたびに State を保存・復元する」グラフになる。
graph_with_memory = builder.compile(checkpointer=memory)

# thread_id = 会話を識別するID。同じIDなら履歴が引き継がれ、違うIDなら別の会話になる。
# (チャットアプリでいう「会話スレッド」「部屋」に相当)
config = {"configurable": {"thread_id": "demo-conversation-001"}}

# 1ターン目: 名前を伝える
result1 = graph_with_memory.invoke(
    {"messages": [{"role": "user", "content": "はじめまして。私の名前は田中です。"}]},
    config,  # ← この config を渡すことで checkpointer が効く
)
print("--- 1ターン目 ---")
result1["messages"][-1].pretty_print()  # [-1] で最後のメッセージ(=最終回答)だけ表示

In [12]:
# 同じ thread_id で 2 ターン目。前のやり取り(自己紹介)が引き継がれているはず
# 「覚えていますか?」と聞くと、Nova Lite は"AIに記憶はない"と前置きしがち。
# 「私の名前で挨拶して」のように 履歴を使うことを促す質問の方が、効果がはっきり見える
result2 = graph_with_memory.invoke(
    {"messages": [{"role": "user", "content": "私の名前で挨拶してください。"}]},
    config,  # 同じ thread_id
)
print("--- 2ターン目 ---")
result2["messages"][-1].pretty_print()

--- 2ターン目 ---
================================== Ai Message ==================================

<thinking>User has asked to be greeted by their name again. I will respond directly to the User.</thinking>
田中さん、こんにちは！お会いできて嬉しいです。何かお手伝いできることがあれば、いつでも私に知らせてください。


In [ ]:
# 異なる thread_id にすると、履歴が共有されない「別の会話」になる。
# 同じ質問をしても、こちらの thread には「田中です」の履歴が無いので名前を知らないはず。
# → thread_id が会話を分離していることが確認できる(履歴は thread ごとに独立)。
result3 = graph_with_memory.invoke(
    {"messages": [{"role": "user", "content": "私の名前で挨拶してください。"}]},
    {"configurable": {"thread_id": "another-conversation"}},  # 別の thread_id
)
print("--- 別 thread(履歴なし)---")
result3["messages"][-1].pretty_print()

## まとめ

- **LangGraph** は AgentExecutor の発展形。`State / Node / Edge` でエージェントの内部構造を明示的に書ける
- `create_react_agent` で素早く動かす → 必要に応じて手書き StateGraph に展開、というステップが王道
- `MessagesState` + `ToolNode` + `tools_condition` の組み合わせが最小 ReAct パターン
- **LangSmith** は環境変数を設定するだけでトレースが自動送信される。デバッグが激変する
- **MemorySaver** + `thread_id` で会話履歴を持てる。本番は永続化版に差し替え

次は [06 RAG深掘り](../06_rag_advanced/) で、第2章で作った素朴な RAG を LangGraph 化しつつ、検索精度を上げていきます。